# EMA parameter sweep

Scan a 2-D grid of ema_fast × ema_slow values against the EMA crossover strategy, compare the results, and visualise where the robust regions are.

**What this notebook produces**
- A table with one row per `(ema_fast, ema_slow)` combination
- Performance metrics (trades, win rate, PnL, Sharpe, drawdown, profit factor)
- Exit-reason breakdown per config (how many trades closed on SL / TP / trailing / etc.)
- Average trade duration per config
- Heatmaps so you can see the *region* of good params, not just the single peak

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
import dataclasses
from collections import Counter

import pandas as pd
import plotly.express as px

from engine.backtester import Backtester
from engine.models import StrategyConfig, ExitReason
from engine.strategies import EMACrossoverStrategy

## 1. Data loader

start_time / end_time accept anything `pandas.Timestamp` parses (ISO strings, datetimes). Naive values are treated as UTC. Leave `END_TIME = None` to fetch up to *now*.

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval
START_TIME, END_TIME = ACTIVE.start, ACTIVE.end

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()
print(f"Loaded {len(df)} bars from {df.index[0]} to {df.index[-1]}")

## 2. Sweep ranges

The full grid is evaluated — `fast >= slow` combinations are **not** skipped. When `fast == slow` the two EMAs are identical and no crosses fire (zero trades). When `fast > slow` the roles invert (the nominally 'fast' EMA is smoother); the strategy still trades, but on a different signal.

In [5]:
EMA_FAST_VALUES = range(5, 31, 2)     # 5, 7, 9, ..., 29
EMA_SLOW_VALUES = range(20, 101, 5)   # 20, 25, 30, ..., 100

n_pairs = len(list(EMA_FAST_VALUES)) * len(list(EMA_SLOW_VALUES))
print(f"{n_pairs} parameter combinations to evaluate")

221 parameter combinations to evaluate


## 3. Run sweep

For each config: build a new frozen StrategyConfig via `dataclasses.replace`, run the backtester, record metrics + per-exit-reason counts + average trade duration (minutes).

In [6]:
rows = []
for fast in EMA_FAST_VALUES:
    for slow in EMA_SLOW_VALUES:
        cfg = dataclasses.replace(StrategyConfig(), ema_fast=fast, ema_slow=slow)
        result = Backtester(EMACrossoverStrategy(cfg), symbol=SYMBOL).run(df, interval=INTERVAL)

        reason_counts = Counter(
            t.exit_reason.value if t.exit_reason else "unknown"
            for t in result.trades
        )
        durations_min = [
            t.duration.total_seconds() / 60
            for t in result.trades if t.duration is not None
        ]
        avg_duration_min = sum(durations_min) / len(durations_min) if durations_min else 0.0

        row = {
            "ema_fast":         fast,
            "ema_slow":         slow,
            "trades":           result.total_trades,
            "win_rate":         result.win_rate,
            "total_pnl_bps":    result.total_pnl_bps,
            "profit_factor":    result.profit_factor,
            "max_dd_bps":       result.max_drawdown_bps,
            "sharpe":           result.sharpe_approx,
            "avg_duration_min": avg_duration_min,
        }
        # One column per exit reason (zero-filled when a reason didn't fire)
        for reason in ExitReason:
            row[f"n_{reason.value}"] = reason_counts.get(reason.value, 0)
        rows.append(row)

results = pd.DataFrame(rows)
print(f"Evaluated {len(results)} configurations")

Evaluated 221 configurations


## 4. Top results

Filter out configs with too few trades before ranking — a high Sharpe on 3 trades is noise, not signal.

In [8]:
MIN_TRADES = 20

top = (
    results[results.trades >= MIN_TRADES]
    .sort_values("total_pnl_bps", ascending=False)
    .head(10)
)
top

,ema_fast,ema_slow,trades,win_rate,total_pnl_bps,profit_factor,max_dd_bps,sharpe,avg_duration_min,n_stop_loss,n_take_profit,n_trailing_stop,n_signal_flip,n_time_stop,n_invalidation,n_force_close
169,23,100,35,0.285714,2.356496,1.002345,329.597438,0.000802,162.428571,0,0,30,5,0,0,0
186,25,100,34,0.235294,-80.340476,0.921790,453.542726,-0.027897,164.117647,0,0,30,4,0,0,0
157,23,40,50,0.240000,-81.727724,0.951839,516.279941,-0.015833,169.500000,0,0,44,6,0,0,0
203,27,100,33,0.212121,-83.825757,0.915630,434.504328,-0.029849,164.090909,0,0,30,3,0,0,0
185,25,95,33,0.242424,-93.060901,0.911216,427.660705,-0.032550,163.636364,0,0,30,3,0,0,0
219,29,95,32,0.218750,-103.439288,0.898324,423.371884,-0.037091,163.593750,0,0,30,2,0,0,0
202,27,95,33,0.212121,-109.901671,0.893388,464.648405,-0.038586,164.545455,0,0,30,3,0,0,0
123,19,40,51,0.215686,-142.315808,0.915902,498.548873,-0.027629,180.000000,0,0,44,7,0,0,0
156,23,35,51,0.215686,-188.570384,0.891789,570.855806,-0.036241,178.529412,0,0,44,7,0,0,0
206,29,30,49,0.224490,-216.592214,0.878211,587.435279,-0.042156,181.224490,0,0,44,5,0,0,0


## 5. Heatmaps

Each cell = one `(ema_fast, ema_slow)` configuration. Look for *regions* of green — a bright cell surrounded by red is usually an overfit outlier; a bright cell with bright neighbours is a robust parameter choice.

In [9]:
grid = results.pivot(index="ema_fast", columns="ema_slow", values="sharpe")
fig = px.imshow(
    grid,
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    aspect="auto",
    labels=dict(x="ema_slow", y="ema_fast", color="Sharpe"),
    title=f"Sharpe — {SYMBOL} {INTERVAL}m ({START_TIME} → {END_TIME or 'now'})",
)
fig.show()

In [10]:
grid = results.pivot(index="ema_fast", columns="ema_slow", values="total_pnl_bps")
fig = px.imshow(
    grid,
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    aspect="auto",
    labels=dict(x="ema_slow", y="ema_fast", color="Total P&L (bps)"),
    title=f"Total P&L — {SYMBOL} {INTERVAL}m",
)
fig.show()

## 6. Exit-reason distribution for the top config

Strategy quality isn't just about PnL — *how* trades close matters. A strategy whose winners all come from take_profit behaves very differently from one relying on signal_flip.

In [ ]:
best = top.iloc[0]
exit_cols = [c for c in results.columns if c.startswith("n_")]
exit_breakdown = best[exit_cols].rename(lambda c: c.removeprefix("n_"))
exit_breakdown = exit_breakdown[exit_breakdown > 0]

fig = px.bar(
    x=exit_breakdown.index,
    y=exit_breakdown.values,
    labels=dict(x="exit reason", y="trade count"),
    title=f"Exit reasons — best config (ema_fast={int(best.ema_fast)}, ema_slow={int(best.ema_slow)})",
)
fig.show()